# ARMA Model Tutorial

This notebook demonstrates how to fit an **AutoRegressive Moving Average (ARMA)** model
using the custom numpy-based `ARMAModel` class from the `timeseries` module.

## What is ARMA?

An ARMA(p, q) model combines:
- **AR(p)** — the value depends on *p* past values (autoregressive part)
- **MA(q)** — the value depends on *q* past forecast errors (moving-average part)

$$X_t = c + \sum_{i=1}^{p} \phi_i X_{t-i} + \sum_{j=1}^{q} \theta_j \varepsilon_{t-j} + \varepsilon_t$$

ARMA assumes the series is **stationary**. If it is not, we first difference the
series (giving an ARIMA model).

## Dataset

We use Indian stock market data from the repository's `sample_data` module.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'testing'))

import numpy as np
import matplotlib.pyplot as plt

from timeseries import moving_average, difference, autocorrelation, ARMAModel
from sample_data import NIFTY50_MONTHLY, MONTHS_2023

## 1. Visualise the data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(MONTHS_2023, NIFTY50_MONTHLY, marker='o')
ax.set_title('Nifty 50 Monthly Close — 2023')
ax.set_ylabel('Index Value')
plt.tight_layout()
plt.show()

## 2. Explore with library helpers

Use the repository's own `moving_average`, `difference`, and `autocorrelation`
functions to understand the series before modelling.

In [ ]:
values = NIFTY50_MONTHLY

# Moving average (window = 3 months)
ma3 = moving_average(values, 3)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(MONTHS_2023, values, alpha=0.5, label='Original')
ax.plot(MONTHS_2023, ma3, label='3-month MA', linewidth=2)
ax.set_title('Nifty 50 with 3-Month Moving Average')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Differencing to make the series stationary
diff1 = difference(values, lag=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(MONTHS_2023[1:], diff1, marker='o')
ax.set_title('First-Order Differenced Nifty 50')
ax.axhline(0, color='grey', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation at various lags
lags = range(1, 6)
acf_values = [autocorrelation(diff1, lag) for lag in lags]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lags, acf_values)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('ACF of Differenced Nifty 50')
plt.tight_layout()
plt.show()

## 3. Fit an ARMA model (numpy-based)

We fit the model on the **differenced** series using the custom `ARMAModel`.

In [ ]:
model = ARMAModel(p=2, q=2)
model.fit(diff1)

print('ARMA(2,2) Fitted Parameters')
print(f'  Intercept : {model.intercept:.4f}')
print(f'  AR coeffs : {np.round(model.ar_coeffs, 4)}')
print(f'  MA coeffs : {np.round(model.ma_coeffs, 4)}')

## 4. Diagnostics — residuals

In [ ]:
resid = model.residuals[max(model.p, model.q):]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(resid)
axes[0].axhline(0, color='grey', linestyle='--')
axes[0].set_title('Residuals')

axes[1].hist(resid, bins=8, density=True, alpha=0.7)
axes[1].set_title('Histogram of Residuals')
plt.tight_layout()
plt.show()

## 5. Forecasting

In [ ]:
n_ahead = 3
fc = model.forecast(steps=n_ahead)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(diff1)), diff1, marker='o', label='Observed (differenced)')
ax.plot(range(len(diff1), len(diff1) + n_ahead), fc, marker='s',
        color='red', label='Forecast')
ax.set_title('ARMA(2,2) — Differenced Nifty 50 Forecast')
ax.legend()
plt.tight_layout()
plt.show()

print('Forecasted differenced values:', [round(v, 2) for v in fc])

## Summary

| Step | Tool / Function |
|---|---|
| Trend inspection | `timeseries.moving_average` |
| Stationarity via differencing | `timeseries.difference` |
| ACF analysis | `timeseries.autocorrelation` |
| ARMA fitting & forecasting | `timeseries.ARMAModel` (numpy) |